# Production-Grade Fine-Tuning Pipeline for Face Recognition
**Architectural Blueprint: AdaFace IR-18 & EdgeFace-S on Kaggle Multi-GPU**

---

### Key Technical Highlights & Pipeline Optimizations
1. **Balanced Stratified Sampling (500,000 Images Target):**
   - Filters out identities with $< 5$ images (eliminates zero-variance classes).
   - Caps images at $\le 30$ per identity (prevents class imbalance and gradient dominance).
   - Collects 500,000 total images across ~17,000-20,000 identities for optimal convergence.
2. **Zero-Disk-Overflow Preprocessing (`SKIP_OFFLINE_CROPPING = True`):**
   - Directly streams pre-aligned 112x112 dataset folders, eliminating Kaggle `Errno 28` disk exhaustion.
   - Includes graceful exception handling for disk write limits.
3. **PyTorch Subset Isolation (`TransformedSubset`):**
   - Prevents augmentation leakage between `train_set` and `val_set` after `random_split`.
4. **Multi-GPU Parallelism & Mixed Precision:**
   - Multi-GPU execution via `nn.DataParallel` with Automatic Mixed Precision (AMP `torch.cuda.amp`).
   - ArcFace Cosine Margin Loss ($\mathbf{s}=64.0, \mathbf{m}=0.50$).
5. **ONNX Export Ready:**
   - One-click export to deployment-ready `.onnx` model with dynamic batching.

## Step 1: Environment Setup & Seed Initialization

In [ ]:
# 1. Install facenet-pytorch cleanly without breaking environment dependencies
!pip install -q --no-deps facenet-pytorch

import os
import sys
import math
import time
import random
import urllib.request
from pathlib import Path
from typing import Tuple, List, Dict, Optional, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
from PIL import Image, ImageFilter
import cv2
import numpy as np
from tqdm.notebook import tqdm

# Pillow compatibility monkey-patch for older torchvision/pillow builds
import PIL._util
if not hasattr(PIL._util, "is_directory"):
    def is_directory(path):
        return isinstance(path, (str, bytes)) and os.path.isdir(path)
    PIL._util.is_directory = is_directory

# Reproducibility Seed Initialization
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    print(f"[INFO] Reproducibility seed set to: {seed}")

set_seed(42)

# Face Detection Backend Check
try:
    from facenet_pytorch import MTCNN
    HAS_MTCNN = True
    print("[INFO] facenet-pytorch MTCNN initialized successfully.")
except Exception:
    HAS_MTCNN = False
    print("[WARNING] facenet-pytorch MTCNN unavailable. OpenCV Haar Cascade will be used as fallback.")

# Compute Infrastructure Inspection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"[INFO] PyTorch Version : {torch.__version__}")
print(f"[INFO] Compute Device   : {device}")
print(f"[INFO] Available GPUs   : {num_gpus}")
for i in range(num_gpus):
    print(f"   └─ GPU [{i}]: {torch.cuda.get_device_name(i)}")

## Step 2: Global Configuration & Hyperparameter Suite

In [ ]:
class Config:
    """Master Hyperparameter & Pipeline Configuration Suite."""
    
    # -------------------------------------------------------------------------
    # 1. Dataset Paths (Automatic Environment Resolution)
    # -------------------------------------------------------------------------
    KAGGLE_DATA_PATH = "/kaggle/input/datasets/kyquac/asian-celeb-112x112/asian_celeb_112x112_folders"
    KAGGLE_ALT_PATH  = "/kaggle/input/asian-celeb-112x112/asian_celeb_112x112_folders"
    LOCAL_DATA_PATH   = r"C:\SCHOOL\asian_celeb_112x112_folders"
    
    if os.path.exists(KAGGLE_DATA_PATH):
        RAW_DATA_DIR = KAGGLE_DATA_PATH
    elif os.path.exists(KAGGLE_ALT_PATH):
        RAW_DATA_DIR = KAGGLE_ALT_PATH
    else:
        RAW_DATA_DIR = LOCAL_DATA_PATH
        
    OUTPUT_DIR = "/kaggle/working/output" if os.path.exists("/kaggle/working") else "./output"
    CROPPED_DATA_DIR = os.path.join(OUTPUT_DIR, "cropped_faces_112x112")
    
    # -------------------------------------------------------------------------
    # 2. Disk & Sampling Controls
    # -------------------------------------------------------------------------
    # Skip offline disk writes (Asian-Celeb dataset is pre-cropped to 112x112)
    SKIP_OFFLINE_CROPPING: bool = True
    
    # Balanced Dataset Sampling Rules (Target: ~500,000 images)
    ENABLE_SAMPLING: bool = True
    TARGET_TOTAL_IMAGES: int = 500000
    MIN_IMAGES_PER_IDENTITY: int = 5    # Exclude single-image / low-sample identities (< 5)
    MAX_IMAGES_PER_IDENTITY: int = 30   # Cap at 30 images/person for balanced class distributions
    
    # -------------------------------------------------------------------------
    # 3. Model & Hyperparameters
    # -------------------------------------------------------------------------
    MODEL_NAME: str = "adaface_ir18"      # Options: 'adaface_ir18', 'edgeface_s'
    EMBEDDING_SIZE: int = 512
    IMAGE_SIZE: int = 112
    
    NUM_GPUS: int = max(1, torch.cuda.device_count())
    CROP_BATCH_SIZE: int = 128 * NUM_GPUS
    BATCH_SIZE: int = 64 * NUM_GPUS
    EPOCHS: int = 15
    LEARNING_RATE: float = 1e-3 * NUM_GPUS
    BACKBONE_LR: float = 1e-4 * NUM_GPUS
    WEIGHT_DECAY: float = 5e-4
    TRAIN_VAL_SPLIT: float = 0.85
    NUM_WORKERS: int = min(4, os.cpu_count() or 4) if os.name != "nt" else 0
    
    # -------------------------------------------------------------------------
    # 4. ArcFace Loss Parameters
    # -------------------------------------------------------------------------
    ARCFACE_SCALE: float = 64.0
    ARCFACE_MARGIN: float = 0.50

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print(f"[INFO] Raw Data Source        : {Config.RAW_DATA_DIR}")
print(f"[INFO] Output Directory        : {Config.OUTPUT_DIR}")
print(f"[INFO] Skip Offline Cropping  : {Config.SKIP_OFFLINE_CROPPING}")
print(f"[INFO] Sampling Mode Active   : {Config.ENABLE_SAMPLING} (Target: {Config.TARGET_TOTAL_IMAGES:,} images)")
print(f"[INFO] Class Filters          : Min {Config.MIN_IMAGES_PER_IDENTITY} | Max {Config.MAX_IMAGES_PER_IDENTITY} images/identity")
print(f"[INFO] Training Workers       : {Config.NUM_WORKERS} | Batch Size: {Config.BATCH_SIZE}")

## Step 3: GPU Batch Preprocessing & Disk Safeguard

In [ ]:
class RawImageBatchDataset(Dataset):
    """Simple raw image loader for GPU batch face cropping."""
    def __init__(self, image_paths: List[Path]):
        self.image_paths = image_paths
        
    def __len__(self) -> int:
        return len(self.image_paths)
        
    def __getitem__(self, idx: int) -> Tuple[Image.Image, str]:
        img_path = self.image_paths[idx]
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (Config.IMAGE_SIZE, Config.IMAGE_SIZE))
        return img, str(img_path)

def preprocess_crop_faces_fast(
    raw_dir: str, 
    target_dir: str, 
    batch_size: int = Config.CROP_BATCH_SIZE, 
    skip_crop: bool = Config.SKIP_OFFLINE_CROPPING
) -> str:
    """Ultra-fast GPU face detector & cropper with disk full exception fallback."""
    if skip_crop:
        print("[INFO] Bypassing offline disk cropping (Dataset is pre-aligned 112x112 faces).")
        print(f"[INFO] Using raw dataset path directly: {raw_dir}")
        return raw_dir
        
    raw_path = Path(raw_dir)
    target_path = Path(target_dir)
    target_path.mkdir(parents=True, exist_ok=True)
    
    print(f"[INFO] Indexing raw images in {raw_dir}...")
    all_img_paths = [p for p in raw_path.glob("*/*") if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]]
    print(f"[INFO] Total images found for cropping: {len(all_img_paths):,}")
    
    if HAS_MTCNN and torch.cuda.is_available():
        mtcnn = MTCNN(image_size=Config.IMAGE_SIZE, margin=14, post_process=False, device=device)
    else:
        mtcnn = None
        cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        detector = cv2.CascadeClassifier(cascade_path)
        
    dataset = RawImageBatchDataset(all_img_paths)
    loader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=Config.NUM_WORKERS, 
        collate_fn=lambda b: ([x[0] for x in b], [x[1] for x in b])
    )
    
    total_saved = 0
    start_time = time.time()
    
    for img_list, path_list in tqdm(loader, desc="GPU Face Cropping"):
        if mtcnn is not None:
            try:
                boxes_batch, probs_batch = mtcnn.detect(img_list)
            except Exception:
                boxes_batch = [None] * len(img_list)
                probs_batch = [None] * len(img_list)
        else:
            boxes_batch = [None] * len(img_list)
            probs_batch = [None] * len(img_list)
            
        for img_pil, raw_path_str, boxes, probs in zip(img_list, path_list, boxes_batch, probs_batch):
            raw_p = Path(raw_path_str)
            out_dir = target_path / raw_p.parent.name
            out_dir.mkdir(exist_ok=True)
            save_path = out_dir / f"{raw_p.stem}_crop.jpg"
            
            if save_path.exists():
                total_saved += 1
                continue
                
            cropped_img = None
            if boxes is not None and len(boxes) > 0:
                best_idx = np.argmax(probs) if probs is not None else 0
                box = boxes[best_idx]
                w, h = img_pil.size
                margin = 14
                x1 = max(0, int(box[0]) - margin)
                y1 = max(0, int(box[1]) - margin)
                x2 = min(w, int(box[2]) + margin)
                y2 = min(h, int(box[3]) + margin)
                if x2 > x1 and y2 > y1:
                    cropped_img = img_pil.crop((x1, y1, x2, y2)).resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE))
                    
            if cropped_img is None:
                w, h = img_pil.size
                min_dim = min(w, h)
                left = (w - min_dim) // 2
                top = (h - min_dim) // 2
                cropped_img = img_pil.crop((left, top, left + min_dim, top + min_dim)).resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE))
                
            try:
                cropped_img.save(save_path)
                total_saved += 1
            except OSError as e:
                if e.errno == 28 or "No space left on device" in str(e):
                    print(f"\n[WARNING] Disk full limit reached at {total_saved:,} files.")
                    print(f"[WARNING] Falling back gracefully to raw directory: {raw_dir}")
                    return raw_dir
                raise e
                
    elapsed = time.time() - start_time
    print(f"[SUCCESS] Cropping finished: {total_saved:,} images in {elapsed:.1f}s")
    return target_dir

DATASET_READY_DIR = preprocess_crop_faces_fast(Config.RAW_DATA_DIR, Config.CROPPED_DATA_DIR)

## Step 4: Dataset Pipeline, Stratified Sampling & Face Augmentations

In [ ]:
class FaceIdentityDataset(Dataset):
    """Production PyTorch Dataset with Identity Filtering & Stratified Sampling."""
    
    def __init__(self, root_dir: str):
        self.root_dir = Path(root_dir)
        self.image_paths: List[Path] = []
        self.labels: List[int] = []
        self.class_to_idx: Dict[str, int] = {}
        self.idx_to_class: Dict[int, str] = {}
        
        all_dirs = [d for d in self.root_dir.iterdir() if d.is_dir()]
        print(f"[INFO] Scanning identity subfolders in {root_dir}...")
        
        valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        
        for cls_dir in all_dirs:
            img_files = [f for f in cls_dir.glob("*.*") if f.suffix.lower() in valid_extensions]
            
            # --- APPLY STRATIFIED SAMPLING & CLASS BALANCING RULES ---
            if Config.ENABLE_SAMPLING:
                # Rule 1: Exclude low-sample identities (< MIN_IMAGES_PER_IDENTITY)
                if len(img_files) < Config.MIN_IMAGES_PER_IDENTITY:
                    continue
                # Rule 2: Cap images per identity to prevent class imbalance (> MAX_IMAGES_PER_IDENTITY)
                if len(img_files) > Config.MAX_IMAGES_PER_IDENTITY:
                    img_files = img_files[:Config.MAX_IMAGES_PER_IDENTITY]
                    
            cls_name = cls_dir.name
            current_idx = len(self.class_to_idx)
            self.class_to_idx[cls_name] = current_idx
            self.idx_to_class[current_idx] = cls_name
            
            for img_file in img_files:
                self.image_paths.append(img_file)
                self.labels.append(current_idx)
                
            # Rule 3: Stop scanning once TARGET_TOTAL_IMAGES is reached
            if Config.ENABLE_SAMPLING and len(self.image_paths) >= Config.TARGET_TOTAL_IMAGES:
                break
                
        print(f"[SUCCESS] Dataset Loaded Successfully!")
        print(f"   ├─ Total Images Loaded     : {len(self.image_paths):,}")
        print(f"   └─ Unique Identities (N)   : {len(self.class_to_idx):,}")
        
    def __len__(self) -> int:
        return len(self.image_paths)
        
    def __getitem__(self, idx: int) -> Tuple[Image.Image, int]:
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (Config.IMAGE_SIZE, Config.IMAGE_SIZE))
        return image, label


class TransformedSubset(Dataset):
    """Wrapper to decouple PyTorch Subset transformation states."""
    
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __len__(self) -> int:
        return len(self.subset)
        
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


# --- SPECIALIZED FACE RECOGNITION AUGMENTATIONS ---
class RandomGaussianBlur(object):
    """Applies Gaussian Blur with probability p."""
    def __init__(self, p: float = 0.25, radius_range: Tuple[float, float] = (0.5, 1.5)):
        self.p = p
        self.radius_range = radius_range
        
    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() < self.p:
            radius = random.uniform(*self.radius_range)
            img = img.filter(ImageFilter.GaussianBlur(radius=radius))
        return img


train_transform = T.Compose([
    T.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=8),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.RandomGrayscale(p=0.10),
    RandomGaussianBlur(p=0.25),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    T.RandomErasing(p=0.25, scale=(0.02, 0.20), ratio=(0.3, 3.3), value="random")
])

val_transform = T.Compose([
    T.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

## Step 5: ArcFace Cosine Angular Margin Loss Head

In [ ]:
class ArcFaceHead(nn.Module):
    """ArcFace: Additive Angular Margin Loss for Deep Face Recognition (Cos Margin Head)."""
    
    def __init__(self, in_features: int, num_classes: int, s: float = 64.0, m: float = 0.50):
        super(ArcFaceHead, self).__init__()
        self.in_features = in_features
        self.num_classes = num_classes
        self.s = s
        self.m = m
        
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m
        
    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        norm_embeddings = F.normalize(embeddings, p=2, dim=1)
        norm_weight = F.normalize(self.weight, p=2, dim=1)
        
        cosine = F.linear(norm_embeddings, norm_weight).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(0, 1)
        
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        
        one_hot = torch.zeros(cosine.size(), device=embeddings.device)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output

## Step 6: Model Architectures & Pretrained Backbone Loader

In [ ]:
def conv3x3(in_planes: int, out_planes: int, stride: int = 1) -> nn.Conv2d:
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

class IResNetBlock(nn.Module):
    """Inverted ResNet Block for Deep Face Recognition."""
    def __init__(self, inplanes: int, planes: int, stride: int = 1):
        super(IResNetBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(inplanes)
        self.conv1 = conv3x3(inplanes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.prelu = nn.PReLU(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn3 = nn.BatchNorm2d(planes)
        self.downsample = None
        if stride != 1 or inplanes != planes:
            self.downsample = nn.Sequential(
                nn.Conv2d(inplanes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )
            
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = self.bn1(x)
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn3(out)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        return out

class IResNet18(nn.Module):
    """IResNet-18 Backbone for AdaFace / EdgeFace Architectures."""
    def __init__(self, embedding_size: int = 512):
        super(IResNet18, self).__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.prelu = nn.PReLU(64)
        
        self.layer1 = self._make_layer(64, 2, stride=2)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        
        self.bn2 = nn.BatchNorm2d(512)
        self.fc = nn.Linear(512 * 7 * 7, embedding_size)
        self.features = nn.BatchNorm1d(embedding_size)
        
    def _make_layer(self, planes: int, blocks: int, stride: int = 1) -> nn.Sequential:
        layers = [IResNetBlock(self.inplanes, planes, stride)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(IResNetBlock(self.inplanes, planes))
        return nn.Sequential(*layers)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.prelu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.bn2(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        x = self.features(x)
        return x

PRETRAINED_URLS = {
    "adaface_ir18": "https://github.com/mk-minchul/AdaFace/releases/download/v1.0/adaface_ir18_ms1mv2.pt",
    "edgeface_s": "https://github.com/MeViLab/EdgeFace/releases/download/v1.0/edgeface_s_gamma_05.pt"
}

def load_pretrained_backbone(model_name: str, embedding_size: int = 512) -> nn.Module:
    """Instantiates backbone and initializes with pretrained weights if available."""
    print(f"[INFO] Initializing Backbone Architecture: {model_name}...")
    model = IResNet18(embedding_size=embedding_size)
    
    weights_dir = Path("./pretrained_weights")
    weights_dir.mkdir(exist_ok=True)
    weight_path = weights_dir / f"{model_name}.pt"
    
    if not weight_path.exists() and model_name in PRETRAINED_URLS:
        url = PRETRAINED_URLS[model_name]
        print(f"[INFO] Downloading pretrained weights from {url}...")
        try:
            urllib.request.urlretrieve(url, weight_path)
            print(f"[SUCCESS] Downloaded weights to {weight_path}")
        except Exception as e:
            print(f"[WARNING] Weight download failed ({e}). Proceeding with random weights.")
            return model
            
    if weight_path.exists():
        print(f"[INFO] Loading checkpoint: {weight_path}")
        try:
            checkpoint = torch.load(weight_path, map_location="cpu")
            state_dict = checkpoint.get("state_dict", checkpoint.get("model", checkpoint))
            cleaned_state_dict = {k.replace("module.", "").replace("backbone.", ""): v for k, v in state_dict.items()}
            model.load_state_dict(cleaned_state_dict, strict=False)
            print("[SUCCESS] Checkpoint loaded smoothly.")
        except Exception as e:
            print(f"[WARNING] Failed to parse checkpoint state_dict: {e}")
            
    return model

## Step 7: Production Multi-GPU Training Loop with Automatic Mixed Precision (AMP)

In [ ]:
def train_model() -> Path:
    """Executes the multi-GPU fine-tuning pipeline with AMP and ArcFace Head."""
    print("=" * 70)
    print(f"🎓 STARTING MULTI-GPU FINE-TUNING: {Config.MODEL_NAME.upper()}")
    print(f"🎓 ACTIVE GPUS: {Config.NUM_GPUS} | TOTAL BATCH SIZE: {Config.BATCH_SIZE}")
    print("=" * 70)
    
    full_dataset = FaceIdentityDataset(DATASET_READY_DIR)
    num_classes = len(full_dataset.class_to_idx)
    
    train_size = int(Config.TRAIN_VAL_SPLIT * len(full_dataset))
    val_size = len(full_dataset) - train_size
    raw_train_set, raw_val_set = random_split(full_dataset, [train_size, val_size])
    
    train_set = TransformedSubset(raw_train_set, transform=train_transform)
    val_set = TransformedSubset(raw_val_set, transform=val_transform)
    
    train_loader = DataLoader(
        train_set, batch_size=Config.BATCH_SIZE, shuffle=True, 
        num_workers=Config.NUM_WORKERS, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_set, batch_size=Config.BATCH_SIZE, shuffle=False, 
        num_workers=Config.NUM_WORKERS, pin_memory=True
    )
    
    # Initialize Backbone and ArcFace Loss Head
    backbone = load_pretrained_backbone(Config.MODEL_NAME, Config.EMBEDDING_SIZE).to(device)
    head = ArcFaceHead(Config.EMBEDDING_SIZE, num_classes, s=Config.ARCFACE_SCALE, m=Config.ARCFACE_MARGIN).to(device)
    
    # Wrap DataParallel for Multi-GPU execution
    if Config.NUM_GPUS > 1:
        print(f"[INFO] Enabling torch.nn.DataParallel across {Config.NUM_GPUS} GPUs.")
        backbone = nn.DataParallel(backbone)
        head = nn.DataParallel(head)
        
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW([
        {"params": backbone.parameters(), "lr": Config.BACKBONE_LR},
        {"params": head.parameters(), "lr": Config.LEARNING_RATE}
    ], weight_decay=Config.WEIGHT_DECAY)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    
    best_val_acc = 0.0
    best_model_path = Path(Config.OUTPUT_DIR) / f"{Config.MODEL_NAME}_best.pt"
    
    for epoch in range(1, Config.EPOCHS + 1):
        start_time = time.time()
        backbone.train()
        head.train()
        
        running_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{Config.EPOCHS:02d} [Train]")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                embeddings = backbone(images)
                outputs = head(embeddings, labels)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data).item()
            total += labels.size(0)
            
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{correct/total*100:.2f}%"})
            
        train_loss = running_loss / total
        train_acc = (correct / total) * 100
        
        # Validation Evaluation
        backbone.eval()
        head.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    embeddings = backbone(images)
                    outputs = head(embeddings, labels)
                    loss = criterion(outputs, labels)
                    
                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels.data).item()
                val_total += labels.size(0)
                
        val_loss /= val_total
        val_acc = (val_correct / val_total) * 100
        scheduler.step()
        
        elapsed = time.time() - start_time
        print(f"Epoch {epoch:02d}/{Config.EPOCHS:02d} [{elapsed:.0f}s] | "
              f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%")
              
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            raw_backbone_state = backbone.module.state_dict() if hasattr(backbone, "module") else backbone.state_dict()
            raw_head_state = head.module.state_dict() if hasattr(head, "module") else head.state_dict()
            
            torch.save({
                "epoch": epoch,
                "backbone": raw_backbone_state,
                "head": raw_head_state,
                "val_acc": val_acc,
                "class_to_idx": full_dataset.class_to_idx
            }, best_model_path)
            print(f"  └─ 🎉 New Best Checkpoint Saved! (Val Acc: {val_acc:.2f}%) -> {best_model_path}")
            
    print("=" * 70)
    print(f"✅ TRAINING COMPLETE! Best Validation Accuracy: {best_val_acc:.2f}%")
    print("=" * 70)
    return best_model_path

best_checkpoint_file = train_model()

## Step 8: ONNX Format Export & Deployment Serialization

In [ ]:
def export_to_onnx(checkpoint_path: Path, output_onnx_filename: Optional[str] = None) -> Path:
    """Exports PyTorch Backbone Checkpoint to Deployment-Ready ONNX Model."""
    if output_onnx_filename is None:
        output_onnx_filename = f"{Config.MODEL_NAME}_finetuned.onnx"
        
    onnx_save_path = Path(Config.OUTPUT_DIR) / output_onnx_filename
    print(f"[INFO] Exporting checkpoint {checkpoint_path} to ONNX...")
    
    model = IResNet18(embedding_size=Config.EMBEDDING_SIZE)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["backbone"])
    model.eval()
    
    dummy_input = torch.randn(1, 3, Config.IMAGE_SIZE, Config.IMAGE_SIZE, dtype=torch.float32)
    
    torch.onnx.export(
        model,
        dummy_input,
        str(onnx_save_path),
        export_params=True,
        opset_version=12,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["embedding"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "embedding": {0: "batch_size"}
        }
    )
    print(f"[SUCCESS] ONNX Model Serialized to: {onnx_save_path}")
    return onnx_save_path

onnx_path = export_to_onnx(best_checkpoint_file)